# HelioGuard Phase 6: SFT warm start (Colab T4)

Run the cells in order. The training cell is also the **Resume** cell: run it again after a disconnect and training continues from the last checkpoint, on Drive or, if Drive is gone or stale, on the Hugging Face Hub.

Registered run (docs/phase6_gates.md, Amendment 3): Qwen2.5-1.5B-Instruct, trainer `peft`, seed 0, 2 epochs (200 steps), final checkpoint evaluated.

**Saving is mirrored.** Every checkpoint and the final outputs go to Drive *and* to a private Hub repo (adapter, state.pt, train_summary.json, train_log.jsonl). After each upload the repo listing is printed and the run stops loudly if anything is missing. Evaluation appends every finished episode to a file at once, uploads every 10 episodes, and skips finished episodes on restart.

Setup once: Colab secrets `HF_TOKEN` (write access; required) and `GITHUB_TOKEN` (only if the repo is private), and a T4 GPU runtime. Nothing secret is stored in this notebook.

In [ ]:
# 1. Fresh clone of the repo (token from the Colab secret, never printed or kept in the remote URL)
import os, subprocess
from google.colab import userdata
REPO = "https://github.com/nithinraj49/helioguard"
!rm -rf /content/helioguard
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None   # public repo: clone without a token
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
subprocess.run(["git", "clone", "-q", url, "/content/helioguard"], check=True)
subprocess.run(["git", "-C", "/content/helioguard", "remote", "set-url", "origin", REPO + ".git"], check=True)
%cd /content/helioguard
!git log --oneline -1

In [ ]:
# 2. Install. The pins are the versions used for the registered run: edit them here (and only here) if your
#    Colab needs different ones. torchao>=0.16.0 was needed for the T4 run.
!pip -q install -e ".[sft]" ortools
!pip -q install "peft==0.21.2" "transformers==5.22.0" "accelerate==1.15.0" "torchao>=0.16.0"

In [ ]:
# 3. Check: versions in use (copy these into the run notes) and the GPU
import importlib.metadata as md, torch
for pkg in ("torch", "transformers", "peft", "accelerate", "torchao", "safetensors"):
    print(f"{pkg:13s} {md.version(pkg)}")
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
from google.colab import userdata
# 4. Mount Drive: checkpoints, traces and the final adapter live here, so they survive disconnects
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints"
OUT = "/content/drive/MyDrive/helioguard/sft_1p5b"
os.makedirs(CKPT, exist_ok=True)

# Hugging Face Hub mirror (private repo; HF_TOKEN comes from the Colab secret and is never printed)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HUB_REPO = "nithinraj49/helioguard-sft-1p5b"   # edit if you use another repo name; it is created private
from verispace_engine.hub import HubSync, restore_checkpoint
hub = HubSync(HUB_REPO)                          # fails here, before any training, if the token or repo is wrong
print("Hub repo ready (private):", HUB_REPO, "| files now:", len(hub.list_files()))

In [ ]:
# 5. SFT traces: copy from Drive if present, else regenerate (deterministic, CPU, train tasks only) and save to Drive
import os, shutil
TRACES = "/content/helioguard/data/sft/train_traces.jsonl"
DRIVE_TRACES = "/content/drive/MyDrive/helioguard/sft_1p5b/train_traces.jsonl"
os.makedirs(os.path.dirname(TRACES), exist_ok=True)
if os.path.exists(DRIVE_TRACES):
    shutil.copy(DRIVE_TRACES, TRACES)
    print("copied traces from Drive")
else:
    !python scripts/generate_sft_traces.py --workers 2 | tail -25
    shutil.copy(TRACES, DRIVE_TRACES)
    print("generated traces and saved them to Drive")
print(sum(1 for _ in open(TRACES)), "traces (expected 800)")

In [ ]:
# 4b. Restore the step-22 checkpoint (or any checkpoint) from a Drive folder or from the Hub repo.
#     Optional: the training cell resumes by itself from whichever of Drive and Hub is further along.
#     Use this cell to put a specific checkpoint in place, e.g. one you copied from the lost run.
#     DRIVE_CKPT_FOLDER: a folder holding state.pt and adapter/ (or a folder holding latest/). Leave "" to use the Hub.
DRIVE_CKPT_FOLDER = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints/latest"
try:
    print(restore_checkpoint(CKPT, drive_dir=DRIVE_CKPT_FOLDER or None, hub=hub))   # Drive folder first, Hub (HUB_REPO) otherwise
except FileNotFoundError as exc:
    print("nothing to restore, training will start from step 0:", exc)
import torch
state = torch.load(f"{CKPT}/latest/state.pt", weights_only=False, map_location="cpu") if os.path.exists(f"{CKPT}/latest/state.pt") else None
print("checkpoint step in state.pt:", state["step"] if state else None)   # expect 22 for the lost run

In [ ]:
# 6. Train / Resume (run this same cell again after a disconnect; it resumes from Drive or the Hub, whichever is further along)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$CKPT" --out-dir "$OUT" --hub-repo "$HUB_REPO"

In [ ]:
# 7. Evaluate the final adapter on the 50 held-out tasks: greedy + temperature 0.7 x 3 seeds, bootstrap CIs, gates (no --greedy-only)
!python -m helioguard.sft_eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter "$OUT/adapter" --out "$OUT/eval" --tag sft_1p5b --hub-repo "$HUB_REPO" --upload-every 10
# Safe to re-run after a disconnect: finished episodes are skipped. Greedy first, then temperature 0.7 seeds 0, 1, 2 (one file each).
# Summarise from the saved files alone (also works on partial runs, no GPU needed):
# !python -m helioguard.sft_eval --model Qwen/Qwen2.5-1.5B-Instruct --out "$OUT/eval" --tag sft_1p5b --aggregate-only

Copy `sft_1p5b_eval_summary.json` and `train_summary.json` from Drive (or from the Hub repo) back into `results/phase6/` in the repo to commit them. Do not commit checkpoints or adapters unless asked.